# 第 1 節｜BERT 遮罩填空與現成情緒分類

本節只做**推論（Inference）**，不訓練模型。請使用 Google Colab GPU（也可用 CPU，但載入較慢）。首次執行需要連線下載模型。

In [ ]:
# Install the required packages for Hugging Face models.
# 安裝執行 Hugging Face 模型所需的套件。
%pip -q install transformers==4.57.1 datasets==3.6.0 accelerate sentencepiece

In [ ]:
# Import all modules used in this notebook.
# 集中匯入這份 Notebook 使用的所有模組。
import torch
from transformers import pipeline

## 1. 預訓練的 BERT：預測 `[MASK]`

模型 ID：`google-bert/bert-base-chinese`。`[MASK]` 代表**一個 Token 的空格**。請注意：這是預測被遮住的文字，**不是情緒分類**。

In [ ]:
# Choose the Hugging Face model ID.
# 指定 Hugging Face 的預訓練模型 ID。
MASK_MODEL_ID = "google-bert/bert-base-chinese"

# Use a GPU when one is available.
# 若有 GPU 就使用 GPU，否則使用 CPU。
device = 0 if torch.cuda.is_available() else -1

# Load the pretrained fill-mask model.
# 載入預訓練的遮罩填空模型。
mask_model = pipeline("fill-mask", model=MASK_MODEL_ID, device=device)

In [ ]:
# Prepare three new sentences with one masked token each.
# 準備三個含有一個遮罩 Token 的新句子。
mask_texts = ["今天天氣真[MASK]。", "她每天認真[MASK]習。", "這個蛋糕非常[MASK]。"]

# Show the three most likely tokens for each sentence.
# 依序顯示每句話機率最高的三個填字候選。
for text in mask_texts:
    # Predict the missing token using its surrounding context.
    # 根據前後文預測被遮住的 Token。
    guesses = mask_model(text, top_k=3)

    # Print the original sentence before its predictions.
    # 先顯示原句，再顯示可能的填字結果。
    print(text)

    # Display candidate tokens with their model scores.
    # 逐一顯示候選文字及其模型分數。
    for guess in guesses:
        print(guess["token_str"], round(guess["score"], 3))

## 2. 已微調的情緒分類模型

模型 ID：`Johnson8187/Chinese-Emotion-Small`。這是以 **mDeBERTa** 為基礎、已針對八類中文情緒微調的模型，**不是原生 BERT，也不是二元正負面模型**。以下直接預測三個新句子。

In [ ]:
# Choose an already fine-tuned emotion model.
# 指定已完成情緒分類微調的模型 ID。
EMOTION_MODEL_ID = "Johnson8187/Chinese-Emotion-Small"

# Load its tokenizer and classification model together.
# 一次載入對應的 Tokenizer 與分類模型。
emotion_model = pipeline("text-classification", model=EMOTION_MODEL_ID, device=device)

In [ ]:
# Prepare the eight emotion labels in the model card order.
# 依照模型卡的編號順序列出八種情緒標籤。
emotion_names = ["平淡語氣", "關切語調", "開心語調", "憤怒語調", "悲傷語調", "疑問語調", "驚奇語調", "厭惡語調"]

# Prepare three new messages for classification.
# 準備三句要進行情緒分類的新留言。
emotion_texts = ["今天拿到獎狀，我真的好開心！", "你最近身體不舒服嗎？我很擔心。", "你怎麼可以故意破壞大家的成果！"]

# Predict labels with the already trained model.
# 使用現成模型預測三句話的情緒標籤。
emotion_results = emotion_model(emotion_texts)

In [ ]:
# Display each message, emotion label, and prediction score.
# 顯示每句留言、預測情緒與分數。
for text, result in zip(emotion_texts, emotion_results):
    # Read the label stored in the model output.
    # 取得模型輸出中的原始標籤。
    label = result["label"]

    # Convert generic LABEL_N values to Chinese when needed.
    # 若標籤為 LABEL_N，就依模型卡轉成中文情緒名稱。
    if label.startswith("LABEL_"):
        label = emotion_names[int(label.replace("LABEL_", ""))]

    # Show only the important prediction information.
    # 僅顯示最重要的預測結果。
    print(text, "→", label, round(result["score"], 3))

**觀察重點：**前者輸出「可能的文字」，後者輸出「八種情緒之一」。分數代表模型的輸出，並不保證判斷正確。試著修改 `MASK_MODEL_ID`、`EMOTION_MODEL_ID` 或三句測試文字。